In [ ]:
import os
import re
import html

import pandas as pd
import requests
from dotenv import load_dotenv

# 프로젝트의 .env 파일에서 API 키 불러오기
load_dotenv("../.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

# 키가 있는지만 확인하고, 실제 값은 출력하지 않음
if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에서 네이버 API 키를 찾을 수 없습니다.")

print("네이버 API 키를 확인했습니다.")

In [ ]:
# NAVER API HUB의 뉴스 검색 주소
url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

# API 인증 정보
headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

# 검색 조건
params = {
    "query": "아모레퍼시픽",
    "display": 10,  # 결과 10건
    "start": 1,     # 첫 번째 결과부터
    "sort": "date", # 최신순
}

# 검색 요청
response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30,
)

# 요청에 문제가 있으면 오류 표시
response.raise_for_status()

# API 응답을 파이썬 데이터로 변환
news_data = response.json()

print("검색 결과 수:", news_data.get("total"))
print("이번에 가져온 기사 수:", len(news_data.get("items", [])))

In [ ]:
# 네이버 검색 결과의 HTML 태그와 특수문자 정리
def clean_news_text(text):
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 검색 결과를 담을 리스트
news_rows = []

for item in news_data.get("items", []):
    news_rows.append({
        "title": clean_news_text(item.get("title")),
        "description": clean_news_text(item.get("description")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "published_at": item.get("pubDate"),
    })

# 리스트를 표로 변환
news_df = pd.DataFrame(news_rows)

print(f"기사 후보: {len(news_df)}건")
display(news_df)

In [ ]:
from bs4 import BeautifulSoup
from markdownify import markdownify
from IPython.display import display, Markdown

# 이전에 확인한 이코노미톡뉴스 기사
article_url = "https://www.economytalk.kr/news/articleView.html?idxno=425107"

# 기사 페이지 가져오기
article_response = requests.get(
    article_url,
    timeout=30,
)
article_response.raise_for_status()

# HTML 읽기
soup = BeautifulSoup(article_response.content, "html.parser")

# 기사 제목과 본문 찾기
title_tag = soup.select_one("h1")
article_tag = soup.select_one("#article-view-content-div")

if article_tag is None:
    raise ValueError("기사 본문을 찾지 못했습니다.")

title = title_tag.get_text(" ", strip=True) if title_tag else "제목 없음"

# 본문을 마크다운으로 변환
article_markdown = markdownify(
    str(article_tag),
    heading_style="ATX",
    table_infer_header=True,
)

print("제목:", title)
display(Markdown(article_markdown))

In [ ]:
import time

# 언론사에서 자주 쓰는 본문 영역
article_selectors = [
    "#article-view-content-div",
    "#article-view-content",
    "#articleBody",
    "#article-body",
    "#newsContent",
    "#newsct_article",
    ".article-body",
    ".article-content",
    ".news_view",
    "article",
]

# 기사별 결과를 담을 리스트
article_results = []

for index, row in news_df.iterrows():
    article_url = row["original_url"]

    # 현재 기사의 결과를 기본값으로 준비
    result = {
        "title": row["title"],
        "original_url": article_url,
        "markdown": None,
        "selector": None,
        "status": "실패",
        "error": None,
    }

    try:
        if not article_url:
            raise ValueError("원문 주소가 없습니다.")

        # 기사 페이지 요청
        article_response = requests.get(article_url, timeout=30)
        article_response.raise_for_status()

        soup = BeautifulSoup(article_response.content, "html.parser")

        # 본문 영역을 순서대로 찾기
        article_tag = None

        for selector in article_selectors:
            candidate = soup.select_one(selector)

            if candidate and len(candidate.get_text(" ", strip=True)) >= 100:
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            raise ValueError("본문 영역을 찾지 못했습니다.")

        # HTML 본문을 마크다운으로 변환
        result["markdown"] = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 본문 후보는 찾았지만 실제 내용 확인이 필요함
        result["status"] = "확인 필요"

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except ValueError as error:
        result["error"] = str(error)

    article_results.append(result)

    print(f"[{index + 1}/{len(news_df)}] {result['status']} · {row['title']}")

    # 사이트에 요청을 연속으로 보내지 않도록 잠시 대기
    time.sleep(1)

# 결과를 표로 변환
articles_df = pd.DataFrame(article_results)

display(
    articles_df[["title", "status", "selector", "error"]]
)

In [ ]:
# 본문을 찾지 못한 기사만 보기
failed_articles = articles_df[
    articles_df["status"] == "실패"
]

display(
    failed_articles[["title", "original_url", "error"]]
)

In [ ]:
# 아시아타임즈 기사 주소
article_url = "https://www.asiatime.co.kr/article/20261105500303"

# 페이지 가져오기
article_response = requests.get(article_url, timeout=30)
article_response.raise_for_status()

soup = BeautifulSoup(article_response.content, "html.parser")

# 아시아타임즈의 본문 영역
article_tag = soup.select_one("#articleContent")

if article_tag is None:
    print("requests로 받은 HTML에서 본문을 찾지 못했습니다.")
else:
    article_markdown = markdownify(
        str(article_tag),
        heading_style="ATX",
        table_infer_header=True,
    )

    print("본문 영역을 찾았습니다!")
    display(Markdown(article_markdown))

In [ ]:
# 아시아타임즈 기사 주소
article_url = "https://www.asiatime.co.kr/article/20261105500303"

# 기존 선택자 목록에 아시아타임즈 본문 위치 추가
if "#articleContent" not in article_selectors:
    article_selectors.insert(0, "#articleContent")

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML에서 본문 찾기
    soup = BeautifulSoup(response.content, "html.parser")
    article_tag = soup.select_one("#articleContent")

    if article_tag is None:
        print("본문을 찾지 못했습니다.")
    else:
        # 본문을 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 articles_df에서 해당 기사 찾기
        article_mask = articles_df["original_url"].str.contains(
            "asiatime.co.kr/article/20261105500303",
            regex=False,
            na=False,
        )

        # 해당 기사의 결과 업데이트
        articles_df.loc[article_mask, "markdown"] = article_markdown
        articles_df.loc[article_mask, "selector"] = "#articleContent"
        articles_df.loc[article_mask, "status"] = "확인 필요"
        articles_df.loc[article_mask, "error"] = None

        print("본문을 찾고 기존 결과를 업데이트했습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("페이지 요청에 실패했습니다:", error)

In [ ]:
# 아직 본문을 찾지 못한 기사만 확인
failed_articles = articles_df[
    articles_df["status"] == "실패"
]

print(f"아직 확인이 필요한 기사: {len(failed_articles)}건")

display(
    failed_articles[["title", "original_url", "error"]]
)

In [ ]:
# 스카이데일리의 본문 선택자 추가
if "#gisaview" not in article_selectors:
    article_selectors.insert(0, "#gisaview")

# 실패했던 8번 기사 주소 가져오기
article_url = articles_df.loc[8, "original_url"]

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML 읽기
    soup = BeautifulSoup(response.content, "html.parser")

    # 스카이데일리의 본문 영역 찾기
    article_tag = soup.select_one("#gisaview")

    if article_tag is None:
        print("본문 영역을 찾지 못했습니다.")
    else:
        # 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 결과 업데이트
        articles_df.at[8, "markdown"] = article_markdown
        articles_df.at[8, "selector"] = "#gisaview"
        articles_df.at[8, "status"] = "확인 필요"
        articles_df.at[8, "error"] = None

        print("스카이데일리 본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

In [ ]:
# 스카이데일리 기사 응답 확인
print("요청한 주소:", article_url)
print("실제로 받은 주소:", response.url)
print("HTTP 상태:", response.status_code)
print("서버가 알려준 인코딩:", response.encoding)

# HTML에 본문 영역 이름이 있는지 확인
print("'gisaview' 포함 여부:", b"gisaview" in response.content)

# 서버가 보낸 HTML 앞부분 확인
print("\n받은 페이지 앞부분:")
print(response.text[:500])

In [ ]:
# 같은 기사의 PC 페이지 주소
desktop_url = "https://www.skyedaily.com/news/news_view.html?ID=314172"

try:
    # PC 페이지 요청
    response = requests.get(desktop_url, timeout=30)
    response.raise_for_status()

    # HTML에 표시된 UTF-8 인코딩 적용
    response.encoding = "utf-8"

    # HTML 읽기
    soup = BeautifulSoup(response.text, "html.parser")

    # PC 페이지의 본문 영역
    article_tag = soup.select_one("#gisaview")

    if article_tag is None:
        print("PC 페이지에서도 본문을 찾지 못했습니다.")
    else:
        # 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 8번 기사의 결과 업데이트
        articles_df.at[8, "markdown"] = article_markdown
        articles_df.at[8, "selector"] = "#gisaview"
        articles_df.at[8, "status"] = "확인 필요"
        articles_df.at[8, "error"] = None

        print("본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

In [ ]:
# PC 페이지에서 받은 원본 바이트
html_bytes = response.content

# 인코딩별로 읽기를 시도하고 본문 일부 확인
for encoding in ["utf-8", "cp949", "euc-kr"]:
    print(f"\n--- {encoding} ---")

    try:
        # 오류를 대체하지 않고 정확히 읽을 수 있는지 확인
        decoded_html = html_bytes.decode(encoding)

        test_soup = BeautifulSoup(decoded_html, "html.parser")
        test_article = test_soup.select_one("#gisaview")

        if test_article is None:
            print("본문 영역을 찾지 못했습니다.")
        else:
            print(test_article.get_text(" ", strip=True)[:200])

    except UnicodeDecodeError:
        print("이 인코딩으로 읽을 수 없습니다.")

In [ ]:
# PC 페이지의 원본 데이터를 cp949로 읽기
decoded_html = response.content.decode("cp949")

# HTML에서 본문 찾기
soup = BeautifulSoup(decoded_html, "html.parser")
article_tag = soup.select_one("#gisaview")

if article_tag is None:
    print("본문 영역을 찾지 못했습니다.")
else:
    # 본문을 마크다운으로 변환
    article_markdown = markdownify(
        str(article_tag),
        heading_style="ATX",
        table_infer_header=True,
    )

    # 8번 기사의 깨진 결과를 정상 내용으로 교체
    articles_df.at[8, "markdown"] = article_markdown
    articles_df.at[8, "selector"] = "#gisaview"
    articles_df.at[8, "status"] = "확인 필요"
    articles_df.at[8, "error"] = None

    print("한글을 정상적으로 읽었습니다!")
    display(Markdown(article_markdown))

In [ ]:
print(articles_df.loc[9, "original_url"])

In [ ]:
# 메트로서울의 본문 선택자 추가
if ".article-txt-contents" not in article_selectors:
    article_selectors.insert(0, ".article-txt-contents")

# 실패했던 9번 기사 주소
article_url = articles_df.loc[9, "original_url"]

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML 읽기
    soup = BeautifulSoup(response.content, "html.parser")

    # 메트로서울의 본문 영역 찾기
    article_tag = soup.select_one(".article-txt-contents")

    if article_tag is None:
        print("본문 영역을 찾지 못했습니다.")
    else:
        # 본문을 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 결과 업데이트
        articles_df.at[9, "markdown"] = article_markdown
        articles_df.at[9, "selector"] = ".article-txt-contents"
        articles_df.at[9, "status"] = "확인 필요"
        articles_df.at[9, "error"] = None

        print("메트로서울 본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

In [ ]:
# 기사별 본문 수집 상태 확인
articles_df["본문_글자수"] = articles_df["markdown"].fillna("").str.len()

display(
    articles_df[
        ["title", "status", "selector", "본문_글자수", "error"]
    ]
)

print("전체 기사:", len(articles_df), "건")
print("본문이 있는 기사:", (articles_df["본문_글자수"] > 0).sum(), "건")
print("본문이 없는 기사:", (articles_df["본문_글자수"] == 0).sum(), "건")

In [ ]:
from urllib.parse import urlparse

# 기사 주소에서 언론사 도메인 꺼내기
articles_df["domain"] = articles_df["original_url"].apply(
    lambda url: urlparse(url).netloc
)

# 언론사별로 사용된 본문 선택자 확인
selector_summary = (
    articles_df[["domain", "selector"]]
    .drop_duplicates()
    .sort_values("domain")
)

display(selector_summary)

In [ ]:
# 스카이데일리는 PC 주소 변환이 필요하므로 우선 별도 처리
general_selectors = selector_summary[
    selector_summary["domain"] != "m.skyedaily.com"
]

# 언론사별로 확인된 선택자를 리스트로 정리
site_selectors = (
    general_selectors
    .groupby("domain")["selector"]
    .apply(list)
    .to_dict()
)

print("일반 본문 선택자")
for domain, selectors in site_selectors.items():
    print(f"{domain}: {selectors}")

print("\n별도 처리")
print("m.skyedaily.com: PC 주소 확인 및 인코딩 처리 필요")

In [ ]:
from urllib.parse import urlparse


def get_article_markdown(url):
    """기사 주소 하나를 받아 본문을 마크다운으로 변환합니다."""

    result = {
        "original_url": url,
        "requested_url": url,
        "selector": None,
        "markdown": None,
        "status": "실패",
        "error": None,
    }

    try:
        domain = urlparse(url).netloc

        # 앞에서 확인한 스카이데일리 기사만 PC 주소로 변경
        is_sky_example = (
            domain == "m.skyedaily.com"
            and "ID=314172" in url
        )

        if is_sky_example:
            result["requested_url"] = (
                "https://www.skyedaily.com/news/news_view.html?ID=314172"
            )

        # 기사 페이지 요청
        response = requests.get(
            result["requested_url"],
            timeout=30,
        )
        response.raise_for_status()

        # 스카이데일리에서 확인한 인코딩 적용
        if is_sky_example:
            page_html = response.content.decode("cp949")
            soup = BeautifulSoup(page_html, "html.parser")

            selectors_to_try = ["#gisaview"]

        else:
            soup = BeautifulSoup(response.content, "html.parser")

            # 해당 언론사에서 확인한 선택자를 먼저 사용
            selectors_to_try = site_selectors.get(domain, [])

        # 언론사 규칙으로 찾지 못하면 공통 선택자도 시도
        selectors_to_try = selectors_to_try + article_selectors

        article_tag = None

        for selector in selectors_to_try:
            candidate = soup.select_one(selector)

            if candidate and candidate.get_text(" ", strip=True):
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 스크립트와 스타일은 본문에서 제외
        for tag in article_tag.select("script, style"):
            tag.decompose()

        # 본문을 마크다운으로 변환
        result["markdown"] = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        result["status"] = "확인 필요"

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except UnicodeDecodeError:
        result["error"] = "페이지의 문자 인코딩을 확인해야 합니다."

    return result

In [ ]:
# 기존에 성공했던 0번 기사로 함수 테스트
test_url = articles_df.loc[0, "original_url"]

test_result = get_article_markdown(test_url)

print("상태:", test_result["status"])
print("사용한 선택자:", test_result["selector"])
print("오류:", test_result["error"])

if test_result["markdown"]:
    display(Markdown(test_result["markdown"]))

In [ ]:
# 테스트할 다른 아시아타임즈 기사 주소를 넣어주세요
test_url = "https://www.asiatime.co.kr/article/20261029500345"

test_result = get_article_markdown(test_url)

print("상태:", test_result["status"])
print("사용한 선택자:", test_result["selector"])
print("오류:", test_result["error"])

if test_result["markdown"]:
    display(Markdown(test_result["markdown"]))

In [ ]:
import time

# 뉴스 검색 API 주소
api_url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

# 받아온 기사 후보를 담을 리스트
all_news_items = []

# 1, 101, 201 ... 901번째 결과부터 각각 100건씩 요청
for start in range(1, 1000, 100):
    params = {
        "query": "아모레퍼시픽",
        "display": 100,
        "start": start,
        "sort": "date",  # 최신순
    }

    response = requests.get(
        api_url,
        headers=headers,
        params=params,
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    items = data.get("items", [])

    all_news_items.extend(items)

    print(
        f"{start}번째부터 {len(items)}건 조회 "
        f"· 누적 {len(all_news_items)}건"
    )

    # 더 이상 결과가 없으면 종료
    if len(items) < 100:
        break

    time.sleep(0.3)

print(f"\n총 받아온 검색 결과: {len(all_news_items)}건")

In [ ]:
# 검색 결과를 표로 정리
all_news_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in all_news_items
])

# 날짜를 한국 시간으로 변환
all_news_df["naver_provided_at"] = pd.to_datetime(
    all_news_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# 2023년 1월 1일 이후부터 현재까지
start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

filtered_news_df = all_news_df[
    all_news_df["naver_provided_at"].between(start_date, end_date)
].copy()

# 같은 원문 주소가 반복된 경우 하나만 남기기
filtered_news_df = filtered_news_df.drop_duplicates(
    subset=["original_url"]
)

print(f"기간과 중복을 정리한 기사 후보: {len(filtered_news_df)}건")
display(filtered_news_df.head(10))


In [ ]:
# 먼저 기사 후보 20건만 선택
test_news_df = filtered_news_df.head(20)

# 원문 수집 결과를 담을 리스트
new_article_results = []

for index, row in test_news_df.iterrows():
    article_url = row["original_url"]

    # 원문 주소가 없는 경우 건너뛰기
    if not isinstance(article_url, str) or not article_url.strip():
        print(f"건너뜀: 원문 주소 없음 · {row['title']}")
        continue

    # 앞에서 만든 함수로 본문 가져오기
    result = get_article_markdown(article_url)

    # 검색 결과의 제목도 함께 기록
    result["title"] = row["title"]
    new_article_results.append(result)

    print(f"{result['status']} · {row['title']}")

    # 요청 사이에 잠시 대기
    time.sleep(1)

# 결과를 표로 변환
new_articles_df = pd.DataFrame(new_article_results)

print(f"\n확인한 기사: {len(new_articles_df)}건")
display(
    new_articles_df[["title", "status", "selector", "error"]]
)

In [ ]:
print(articles_df.loc[1, "original_url"])

In [ ]:
import hashlib
from urllib.parse import urlparse

# 뉴스 스키마로 변환한 결과를 담을 리스트
news_records = []

for _, row in articles_df.iterrows():
    article_url = row["original_url"]
    article_markdown = row["markdown"]

    # 원문 주소가 없는 기사는 건너뛰기
    if not isinstance(article_url, str) or not article_url.strip():
        print("원문 주소가 없어 건너뜀:", row["title"])
        continue

    # URL을 이용해 기사마다 고유한 문서 ID 생성
    url_hash = hashlib.sha256(
        article_url.encode("utf-8")
    ).hexdigest()[:16]

    # 기사 본문이 없는 경우 None으로 처리
    if not isinstance(article_markdown, str) or not article_markdown.strip():
        article_markdown = None

    # 스키마에 맞는 기사 한 건
    news_record = {
        "document_id": f"NEWS-{url_hash}",
        "title": row["title"],

        # 언론사 이름과 원문 발행일은 원문에서 확인 후 입력
        "source_name": None,
        "original_url": article_url,
        "published_at": None,

        # 실제 수집 시각을 기록하지 않았으므로 비워둠
        "collected_at": None,

        # 검색한 기업이지만 관련성은 아직 확인 전
        "company_relations": [
            {
                "company_id": "COMPANY-AMOREPACIFIC",
                "relevance": "판단 어려움",
                "review_status": "검토 필요",
            }
        ],

        # 주제 라벨은 이후 분류
        "topics": [],

        # 추출한 기사 본문
        "content": article_markdown,
        "content_type": "원문 추출" if article_markdown else None,

        # 기사별 이용 조건은 아직 확인 전
        "usage_status": "확인 필요",
    }

    news_records.append(news_record)

# 스키마 표로 변환
news_records_df = pd.DataFrame(news_records)

print(f"스키마에 담은 기사 후보: {len(news_records_df)}건")

display(
    news_records_df[
        [
            "document_id",
            "title",
            "original_url",
            "company_relations",
            "topics",
            "usage_status",
        ]
    ]
)

In [ ]:
# content를 포함해서 표시
display(
    news_records_df[
        ["document_id", "title", "content"]
    ]
)

In [ ]:
display(Markdown(news_records_df.loc[0, "content"]))

In [ ]:
import re

def remove_images(content):
    """마크다운과 HTML의 이미지 표시를 제거합니다."""

    if not isinstance(content, str):
        return content

    # 마크다운 이미지: ![설명](이미지주소)
    content = re.sub(
        r"!\[[^\]]*\]\((?:[^()]|\([^()]*\))*\)",
        "",
        content,
    )

    # HTML 이미지: <img ...>
    content = re.sub(
        r"<img\b[^>]*>",
        "",
        content,
        flags=re.IGNORECASE,
    )

    return content


# 뉴스 데이터의 content에서 이미지 제거
news_records_df["content"] = news_records_df["content"].apply(
    remove_images
)

print("기사 본문에서 이미지 표시를 제거했습니다.")

In [ ]:
# 전체 컬럼과 본문을 생략하지 않고 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(news_records_df)

In [ ]:
# 연도별 검색어
year_queries = [
    "아모레퍼시픽 2023",
    "아모레퍼시픽 2024",
    "아모레퍼시픽 2025",
    "아모레퍼시픽 2026",
]

# 연도별 검색 결과를 담을 리스트
year_news_items = []

for query in year_queries:
    print(f"\n검색 중: {query}")

    # 검색어마다 우선 최대 300건씩 가져오기
    for start in [1, 101, 201]:
        params = {
            "query": query,
            "display": 100,
            "start": start,
            "sort": "sim",  # 검색어 관련도순
        }

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        year_news_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        # 더 이상 검색 결과가 없으면 다음 검색어로 이동
        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n연도별 검색 결과: {len(year_news_items)}건")

In [ ]:
# 연도별 검색 결과를 표로 정리
year_news_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in year_news_items
])

# 날짜를 한국 시간으로 변환
year_news_df["naver_provided_at"] = pd.to_datetime(
    year_news_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# 앞에서 수집한 최신 기사와 합치기
amore_candidates_df = pd.concat(
    [filtered_news_df, year_news_df],
    ignore_index=True,
)

# 2023년 1월 1일부터 현재까지 선택
start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

amore_candidates_df = amore_candidates_df[
    amore_candidates_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
].copy()

# 원문 주소가 없는 결과는 제외
amore_candidates_df = amore_candidates_df[
    amore_candidates_df["original_url"].notna()
    & amore_candidates_df["original_url"].str.strip().ne("")
]

# 같은 원문 주소는 하나만 남기기
amore_candidates_df = amore_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print(f"중복 제거 후 기사 후보: {len(amore_candidates_df)}건")
display(amore_candidates_df.head(10))

In [ ]:
year_counts = (
    amore_candidates_df["naver_provided_at"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("연도별 기사 후보 수")
display(year_counts)

In [ ]:
# 네이버 검색 결과의 기사 주소와 제공 시각 가져오기
news_dates = news_df[
    ["original_url", "published_at"]
].drop_duplicates(subset=["original_url"])

# 기사 주소를 기준으로 날짜 연결
date_by_url = news_dates.set_index("original_url")["published_at"]

news_records_df["published_at"] = (
    news_records_df["original_url"]
    .map(date_by_url)
)

# 날짜를 한국 시간으로 변환
published_dates = pd.to_datetime(
    news_records_df["published_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# JSON에 저장하기 쉬운 문자열로 변환
news_records_df["published_at"] = published_dates.apply(
    lambda value: value.isoformat() if pd.notna(value) else None
)

# 사용하지 않기로 한 컬럼 제거
news_records_df = news_records_df.drop(
    columns=[
        "source_name",
        "collected_at",
        "content_type",
        "usage_status",
    ],
    errors="ignore",
)

# 날짜가 들어갔는지 확인
display(
    news_records_df[["title", "published_at"]]
)

print(
    "날짜 확인이 필요한 기사:",
    news_records_df["published_at"].isna().sum(),
    "건",
)

In [ ]:
# 전체 컬럼과 본문 내용을 생략하지 않고 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(news_records_df)

print(f"전체 기사 수: {len(news_records_df)}건")
print(f"스키마 컬럼 수: {len(news_records_df.columns)}개")

In [ ]:
# 기사 단위 스키마에서 topics 제거
news_records_df = news_records_df.drop(
    columns=["topics"],
    errors="ignore",
)

# 변경된 스키마 확인
print("현재 스키마 컬럼:")
print(news_records_df.columns.tolist())

display(news_records_df)

In [ ]:
import hashlib
import time

# 시도한 기사와 실패 기록을 보관할 변수
# 이미 만들어져 있다면 기존 기록을 유지합니다.
if "attempted_urls" not in globals():
    attempted_urls = set()

if "collection_errors" not in globals():
    collection_errors = []

# 이미 스키마에 들어간 기사 주소
existing_urls = set(
    news_records_df["original_url"].dropna()
)

# 아직 저장하지 않았고, 이번 작업에서 시도하지 않은 기사
remaining_news_df = amore_candidates_df[
    ~amore_candidates_df["original_url"].isin(
        existing_urls | attempted_urls
    )
].copy()

# 이번에 처리할 기사 수
batch_size = 20
batch_df = remaining_news_df.head(batch_size)

print(f"남은 기사 후보: {len(remaining_news_df)}건")
print(f"이번에 처리할 기사: {len(batch_df)}건")

In [ ]:
# 이번에 성공한 기사를 담을 리스트
new_records = []

for number, (_, row) in enumerate(batch_df.iterrows(), start=1):
    article_url = row["original_url"]
    title = row["title"]

    # 같은 기사에 반복 요청하지 않도록 시도 기록
    attempted_urls.add(article_url)

    try:
        # 앞에서 만든 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)

        # 본문을 찾지 못하면 실패 목록에 남기고 다음 기사로 이동
        if not result.get("markdown"):
            collection_errors.append({
                "title": title,
                "original_url": article_url,
                "error": result.get("error"),
            })

            print(f"[{number}/{len(batch_df)}] 실패 · {title}")
            continue

        # 이미지 표시 제거
        content = remove_images(result["markdown"])

        # URL을 이용해 고유 문서 ID 생성
        url_hash = hashlib.sha256(
            article_url.encode("utf-8")
        ).hexdigest()[:16]

        # 네이버가 제공한 시각을 문자열로 변환
        provided_at = row["naver_provided_at"]
        published_at = (
            provided_at.isoformat()
            if pd.notna(provided_at)
            else None
        )

        # 현재 스키마에 맞게 기사 한 건 정리
        record = {
            "document_id": f"NEWS-{url_hash}",
            "title": title,
            "original_url": article_url,
            "published_at": published_at,
            "company_relations": [
                {
                    "company_id": "COMPANY-AMOREPACIFIC",
                    "relevance": "판단 어려움",
                    "review_status": "검토 필요",
                }
            ],
            "content": content,
        }

        new_records.append(record)
        print(f"[{number}/{len(batch_df)}] 본문 후보 확인 · {title}")

    except Exception as error:
        # 예상하지 못한 오류가 나더라도 다음 기사로 진행
        collection_errors.append({
            "title": title,
            "original_url": article_url,
            "error": f"{type(error).__name__}: {error}",
        })

        print(f"[{number}/{len(batch_df)}] 오류 · {title}")

    finally:
        # 언론사에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

# 기존 뉴스 표에 이번 결과 추가
if new_records:
    new_records_df = pd.DataFrame(new_records)

    news_records_df = pd.concat(
        [news_records_df, new_records_df],
        ignore_index=True,
    )

print("\n이번에 추가한 기사:", len(new_records), "건")
print("현재 스키마에 담긴 기사:", len(news_records_df), "건")
print("누적 실패 기사:", len(collection_errors), "건")

In [ ]:
# 정리 전 기사 수
before_count = len(news_records_df)

# 본문이 있는 기사만 남기기
news_records_df = news_records_df[
    news_records_df["content"].apply(
        lambda content: isinstance(content, str) and bool(content.strip())
    )
].reset_index(drop=True)

print(f"제외된 기사: {before_count - len(news_records_df)}건")
print(f"남은 기사: {len(news_records_df)}건")


In [ ]:
news_records_df.head(28)

In [ ]:
# company_relations 컬럼 제거
news_records_df = news_records_df.drop(
    columns=["company_relations"],
    errors="ignore",
)

# 모든 컬럼과 본문을 생략하지 않고 표로 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(news_records_df)

print("현재 기사 수:", len(news_records_df), "건")
print("현재 스키마:", news_records_df.columns.tolist())

In [ ]:
import hashlib
import time

# 기존 시도 기록이 없으면 새로 만들기
if "attempted_urls" not in globals():
    attempted_urls = set()

# 기존 뉴스 표의 인덱스 정리
news_records_df = news_records_df.reset_index(drop=True)

# 이미 담긴 기사 주소
existing_urls = set(
    news_records_df["original_url"].dropna()
)

# 원문 주소가 있는 후보만 선택
valid_candidates = amore_candidates_df[
    amore_candidates_df["original_url"].notna()
].copy()

valid_candidates = valid_candidates[
    valid_candidates["original_url"].str.strip().ne("")
]

# 이미 담았거나 시도한 기사는 제외
remaining_news_df = valid_candidates[
    ~valid_candidates["original_url"].isin(
        existing_urls | attempted_urls
    )
].copy()

success_count = 0
failure_count = 0

print(f"처리할 기사 후보: {len(remaining_news_df)}건")

for number, (_, row) in enumerate(remaining_news_df.iterrows(), start=1):
    article_url = row["original_url"]
    title = row["title"]

    # 이미 시도한 기사는 다시 요청하지 않도록 기록
    attempted_urls.add(article_url)

    try:
        # 앞에서 만든 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)
        article_markdown = result.get("markdown")

        # 본문을 가져오지 못하면 제외
        if not isinstance(article_markdown, str) or not article_markdown.strip():
            failure_count += 1
            continue

        # 이미지 제거
        content = remove_images(article_markdown)

        if not content.strip():
            failure_count += 1
            continue

        # 기사 주소로 고유 문서 ID 생성
        url_hash = hashlib.sha256(
            article_url.encode("utf-8")
        ).hexdigest()[:16]

        # 네이버 제공 시각
        provided_at = row["naver_provided_at"]
        published_at = (
            provided_at.isoformat()
            if pd.notna(provided_at)
            else None
        )

        # 현재 스키마에 맞게 저장
        news_records_df.loc[len(news_records_df)] = {
            "document_id": f"NEWS-{url_hash}",
            "title": title,
            "original_url": article_url,
            "published_at": published_at,
            "content": content,
        }

        success_count += 1

    except Exception:
        # 오류가 발생한 기사는 제외하고 다음 기사로 이동
        failure_count += 1

    finally:
        # 사이트에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

    # 20건마다 진행 상황 표시
    if number % 20 == 0:
        print(
            f"{number}/{len(remaining_news_df)}건 처리 · "
            f"추가 {success_count}건 · 제외 {failure_count}건"
        )

print("\n수집 완료")
print("이번에 추가한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("현재 전체 기사:", len(news_records_df), "건")

In [ ]:
# 기사 수와 스키마 확인
print("전체 기사 수:", len(news_records_df), "건")
print("스키마:", news_records_df.columns.tolist())

# 앞에서 10건을 본문 생략 없이 표로 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(news_records_df.head(10))

In [ ]:
# None, NaN 확인
missing_mask = news_records_df.isna()

# 빈 문자열이나 공백만 있는 값도 확인
blank_mask = news_records_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

# 두 조건 합치기
missing_mask = missing_mask | blank_mask

# 컬럼별 빈값 개수
missing_summary = pd.DataFrame({
    "빈값 수": missing_mask.sum(),
})

print(f"전체 기사: {len(news_records_df)}건")
display(missing_summary)

# 빈값이 하나라도 있는 기사 확인
missing_rows = news_records_df[missing_mask.any(axis=1)]

print(f"빈값이 있는 기사: {len(missing_rows)}건")
display(missing_rows)

In [ ]:
from pathlib import Path
import json
import os
import tempfile

# 저장 경로
output_path = Path(
    "../data/collected/anna/news.jsonl"
)

# 최종 스키마 컬럼
news_columns = [
    "document_id",
    "title",
    "original_url",
    "published_at",
    "content",
]

# 필요한 컬럼이 모두 있는지 확인
missing_columns = [
    column
    for column in news_columns
    if column not in news_records_df.columns
]

if missing_columns:
    raise ValueError(f"스키마에 없는 컬럼: {missing_columns}")

# 최종 컬럼만 선택
output_df = news_records_df[news_columns].copy()

if output_df.empty:
    raise ValueError("저장할 기사 데이터가 없습니다.")

# 빈값 확인
missing_values = output_df.isna() | output_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

if missing_values.any().any():
    raise ValueError("빈값이 있습니다. 저장 전에 확인해 주세요.")

# 문서 ID와 원문 주소 중복 확인
if output_df["document_id"].duplicated().any():
    raise ValueError("중복된 document_id가 있습니다.")

if output_df["original_url"].duplicated().any():
    raise ValueError("중복된 original_url이 있습니다.")

# JSONL 문자열로 변환
jsonl_text = output_df.to_json(
    orient="records",
    lines=True,
    force_ascii=False,
)

# 각 줄이 정상적인 JSON인지 확인
records = [
    json.loads(line)
    for line in jsonl_text.splitlines()
    if line.strip()
]

if len(records) != len(output_df):
    raise ValueError("저장할 데이터의 행 수가 일치하지 않습니다.")

# 임시 파일에 먼저 쓴 뒤 대상 파일 교체
output_path.parent.mkdir(parents=True, exist_ok=True)

with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=output_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)
    temp_file.write(jsonl_text)

try:
    os.replace(temp_path, output_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

print("저장 완료:", output_path)
print("저장된 기사:", len(records), "건")

In [ ]:
# 이니스프리 뉴스 검색
innisfree_params = {
    "query": "이니스프리",
    "display": 10,
    "start": 1,
    "sort": "date",
}

response = requests.get(
    "https://naverapihub.apigw.ntruss.com/search/v1/news",
    headers=headers,
    params=innisfree_params,
    timeout=30,
)
response.raise_for_status()

innisfree_data = response.json()

print("검색 결과 수:", innisfree_data.get("total"))
print("이번에 가져온 기사 수:", len(innisfree_data.get("items", [])))

In [ ]:
import time

# 이니스프리 검색어와 정렬 방식
innisfree_queries = [
    ("이니스프리", "date", 10),       # 최신순 최대 1,000건
    ("이니스프리 2023", "sim", 3),   # 관련도순 최대 300건
    ("이니스프리 2024", "sim", 3),
    ("이니스프리 2025", "sim", 3),
    ("이니스프리 2026", "sim", 3),
]

innisfree_items = []

for query, sort, max_pages in innisfree_queries:
    print(f"\n검색 중: {query}")

    for page in range(max_pages):
        start = 1 + page * 100

        params = {
            "query": query,
            "display": 100,
            "start": start,
            "sort": sort,
        }

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        innisfree_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n가져온 검색 결과: {len(innisfree_items)}건")

In [ ]:
# 검색 결과를 표로 정리
innisfree_search_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in innisfree_items
])

# 네이버 제공 시각을 한국 시간으로 변환
innisfree_search_df["naver_provided_at"] = pd.to_datetime(
    innisfree_search_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

# 기간 안에 있고 원문 주소가 있는 기사만 선택
innisfree_candidates_df = innisfree_search_df[
    innisfree_search_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
    & innisfree_search_df["original_url"].notna()
    & innisfree_search_df["original_url"].str.strip().ne("")
].copy()

# 같은 원문 주소는 하나만 남기기
innisfree_candidates_df = innisfree_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print("중복 제거 후 이니스프리 기사 후보:", len(innisfree_candidates_df), "건")

# 연도별 후보 수 확인
year_counts = (
    innisfree_candidates_df["naver_provided_at"]
    .dt.year
    .value_counts()
    .sort_index()
)

display(year_counts)
display(innisfree_candidates_df.head(10))

In [ ]:
import hashlib
import time

# 이니스프리 기사 결과가 아직 없으면 새로 만들기
if "innisfree_news_df" not in globals():
    innisfree_news_df = pd.DataFrame(
        columns=[
            "document_id",
            "title",
            "original_url",
            "published_at",
            "content",
        ]
    )

# 이미 시도한 기사 주소 기록
if "innisfree_attempted_urls" not in globals():
    innisfree_attempted_urls = set()

# 아직 시도하지 않은 기사만 선택
innisfree_remaining_df = innisfree_candidates_df[
    ~innisfree_candidates_df["original_url"].isin(
        innisfree_attempted_urls
    )
].copy()

success_count = 0
failure_count = 0

print(f"처리할 이니스프리 기사 후보: {len(innisfree_remaining_df)}건")

for number, (_, row) in enumerate(
    innisfree_remaining_df.iterrows(),
    start=1,
):
    article_url = row["original_url"]
    title = row["title"]

    # 이미 시도한 기사는 다시 요청하지 않도록 기록
    innisfree_attempted_urls.add(article_url)

    try:
        # 기존 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)
        article_markdown = result.get("markdown")

        if not isinstance(article_markdown, str) or not article_markdown.strip():
            # 본문을 가져오지 못한 기사는 제외
            failure_count += 1

        else:
            # 이미지 표시 제거
            content = remove_images(article_markdown)

            if not content.strip():
                failure_count += 1

            else:
                # URL을 이용해 문서 ID 생성
                url_hash = hashlib.sha256(
                    article_url.encode("utf-8")
                ).hexdigest()[:16]

                provided_at = row["naver_provided_at"]

                # 최종 뉴스 스키마에 맞게 정리
                record = {
                    "document_id": f"NEWS-{url_hash}",
                    "title": title,
                    "original_url": article_url,
                    "published_at": (
                        provided_at.isoformat()
                        if pd.notna(provided_at)
                        else None
                    ),
                    "content": content,
                }

                innisfree_news_df.loc[len(innisfree_news_df)] = record
                success_count += 1

    except Exception:
        # 오류가 난 기사는 제외하고 다음 기사로 이동
        failure_count += 1

    finally:
        # 사이트에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

    # 20건마다 진행 상황 표시
    if number % 20 == 0:
        print(
            f"{number}/{len(innisfree_remaining_df)}건 처리 · "
            f"추가 {success_count}건 · 제외 {failure_count}건"
        )

print("\n수집 완료")
print("이번에 추가한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("이니스프리 전체 기사:", len(innisfree_news_df), "건")

In [ ]:
# None, NaN 확인
missing_mask = innisfree_news_df.isna()

# 빈 문자열이나 공백만 있는 값 확인
blank_mask = innisfree_news_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

# 두 조건 합치기
missing_mask = missing_mask | blank_mask

# 컬럼별 빈값 개수
missing_summary = pd.DataFrame({
    "빈값 수": missing_mask.sum(),
})

print("전체 기사:", len(innisfree_news_df), "건")
display(missing_summary)

# 빈값이 하나라도 있는 기사
missing_rows = innisfree_news_df[
    missing_mask.any(axis=1)
]

print("빈값이 있는 기사:", len(missing_rows), "건")
display(missing_rows)

In [ ]:
from pathlib import Path
import json
import os
import tempfile

# 기존 뉴스 파일
news_path = Path(
    "../data/collected/anna/news.jsonl"
)

# 기존 JSONL 읽기
with news_path.open(encoding="utf-8") as file:
    news_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

if not news_records:
    raise ValueError("뉴스 파일에 데이터가 없습니다.")

# 모든 기사에 아모레퍼시픽 기업 ID 추가
for record in news_records:
    record["company_ids"] = ["COMPANY-AMOREPACIFIC"]

# 임시 파일에 먼저 쓴 뒤 기존 파일 교체
with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=news_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)

    for record in news_records:
        temp_file.write(json.dumps(record, ensure_ascii=False) + "\n")

try:
    os.replace(temp_path, news_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

# 노트북의 아모레퍼시픽 표도 저장된 내용과 맞추기
news_records_df = pd.DataFrame(news_records)

print("업데이트 완료:", news_path)
print("아모레퍼시픽 ID가 추가된 기사:", len(news_records_df), "건")

display(
    news_records_df[["document_id", "title", "company_ids"]].head(10)
)

In [ ]:
import pandas as pd

# 저장된 뉴스 JSONL 불러오기
news_path = "../data/collected/anna/news.jsonl"

news_records_df = pd.read_json(
    news_path,
    lines=True,
    convert_dates=False,
)

print("불러온 기사 수:", len(news_records_df), "건")
print("컬럼:", news_records_df.columns.tolist())

# 기사 10건 확인
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 100,
):
    display(news_records_df.head(10))

In [ ]:
from pathlib import Path
import json
import os
import tempfile

# 이니스프리 수집 결과가 메모리에 있는지 확인
if "innisfree_news_df" not in globals():
    raise ValueError(
        "이니스프리 수집 결과가 메모리에 없습니다. "
        "별도로 저장해 둔 파일이 있는지 먼저 확인해 주세요."
    )

if innisfree_news_df.empty:
    raise ValueError("이니스프리 기사 데이터가 비어 있습니다.")

news_path = Path(
    "../data/collected/anna/news.jsonl"
)

# 기존 아모레퍼시픽 기사 읽기
with news_path.open(encoding="utf-8") as file:
    existing_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

# 원문 주소로 기존 기사 찾기
records_by_url = {
    record["original_url"]: record
    for record in existing_records
}

# 이니스프리 표를 JSON 객체 목록으로 변환
innisfree_records = json.loads(
    innisfree_news_df.to_json(
        orient="records",
        date_format="iso",
        force_ascii=False,
    )
)

added_count = 0
linked_count = 0

for record in innisfree_records:
    article_url = record.get("original_url")

    if not article_url:
        continue

    if article_url in records_by_url:
        # 이미 있는 기사라면 기업 ID만 추가
        existing = records_by_url[article_url]
        company_ids = existing.get("company_ids") or []

        if "COMPANY-INNISFREE" not in company_ids:
            company_ids.append("COMPANY-INNISFREE")
            existing["company_ids"] = company_ids
            linked_count += 1

    else:
        # 새로운 기사라면 이니스프리 ID를 넣어 추가
        record["company_ids"] = ["COMPANY-INNISFREE"]
        existing_records.append(record)
        records_by_url[article_url] = record
        added_count += 1

# 임시 파일에 쓴 뒤 기존 JSONL 교체
with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=news_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)

    for record in existing_records:
        temp_file.write(json.dumps(record, ensure_ascii=False) + "\n")

try:
    os.replace(temp_path, news_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

# 노트북 표도 저장된 데이터와 맞추기
news_records_df = pd.DataFrame(existing_records)

print("새로 추가한 이니스프리 기사:", added_count, "건")
print("기존 기사에 이니스프리 ID를 추가한 건수:", linked_count, "건")
print("저장된 전체 기사:", len(news_records_df), "건")
print("저장 위치:", news_path)

In [ ]:
import os
import re
import html
import time

from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# API 키 불러오기
load_dotenv("../.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에서 네이버 API 키를 찾을 수 없습니다.")

headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

print("네이버 API 준비 완료")

In [ ]:
def clean_news_text(text):
    """검색 결과의 HTML 태그와 특수문자 정리"""
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 최신순 검색과 연도별 검색
current_year = pd.Timestamp.now(tz="Asia/Seoul").year

innisfree_queries = [("이니스프리", "date", 10)]

for year in range(2023, current_year + 1):
    innisfree_queries.append(
        (f"이니스프리 {year}", "sim", 3)
    )

innisfree_items = []

for query, sort, max_pages in innisfree_queries:
    print(f"\n검색 중: {query}")

    for page in range(max_pages):
        start = 1 + page * 100

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params={
                "query": query,
                "display": 100,
                "start": start,
                "sort": sort,
            },
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        innisfree_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n가져온 검색 결과: {len(innisfree_items)}건")

In [ ]:
# 검색 결과를 표로 변환
innisfree_candidates_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in innisfree_items
])

# 네이버 제공 시각을 한국 시간으로 변환
innisfree_candidates_df["naver_provided_at"] = pd.to_datetime(
    innisfree_candidates_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

# 기간 안에 있고 원문 주소가 있는 기사만 남기기
innisfree_candidates_df = innisfree_candidates_df[
    innisfree_candidates_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
    & innisfree_candidates_df["original_url"].notna()
    & innisfree_candidates_df["original_url"].str.strip().ne("")
].copy()

# 같은 원문 주소 제거
innisfree_candidates_df = innisfree_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print("중복 제거 후 기사 후보:", len(innisfree_candidates_df), "건")

# 후보 목록을 별도 파일로 저장
candidates_path = Path(
    "../private_data/innisfree_candidates.jsonl"
)

if candidates_path.exists():
    raise FileExistsError(
        f"이미 후보 파일이 있습니다. 덮어쓰지 않았습니다: {candidates_path}"
    )

with candidates_path.open("x", encoding="utf-8") as file:
    file.write(
        innisfree_candidates_df.to_json(
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    )

print("후보 목록 저장 완료:", candidates_path)

In [ ]:
import hashlib
import json
import time

from bs4 import BeautifulSoup
from markdownify import markdownify
from urllib.parse import urlparse


# 앞에서 확인한 언론사별 본문 위치
site_selectors = {
    "www.asiatime.co.kr": ["#articleContent"],
    "www.metroseoul.co.kr": [".article-txt-contents"],
    "biz.chosun.com": [".article-body"],
    "www.etnews.com": ["#articleBody"],
    "www.techholic.co.kr": ["#articleBody"],
    "www.jejudomin.co.kr": ["#article-view-content-div"],
    "www.newsian.co.kr": ["#article-view-content-div"],
    "www.pinpointnews.co.kr": ["#article-view-content-div"],
    "www.newsis.com": ["article"],
}

# 사이트별 규칙으로 찾지 못했을 때 시도할 위치
common_selectors = [
    "#article-view-content-div",
    "#articleContent",
    ".article-txt-contents",
    "#articleBody",
    "#article-body",
    "#newsContent",
    ".article-body",
    ".article-content",
    ".news_view",
    "article",
]


def get_article_markdown(url):
    """기사 주소에서 본문 후보를 찾아 마크다운으로 변환합니다."""

    result = {
        "markdown": None,
        "error": None,
    }

    try:
        domain = urlparse(url).netloc.lower()
        request_url = url

        # 이전에 확인한 스카이데일리 기사에만 적용
        is_sky_example = (
            domain == "m.skyedaily.com"
            and "ID=314172" in url
        )

        if is_sky_example:
            request_url = (
                "https://www.skyedaily.com/news/news_view.html?ID=314172"
            )

        response = requests.get(request_url, timeout=30)
        response.raise_for_status()

        # 해당 스카이데일리 기사에서는 cp949 사용
        if is_sky_example:
            page_html = response.content.decode("cp949")
            soup = BeautifulSoup(page_html, "html.parser")
            selectors = ["#gisaview"]
        else:
            soup = BeautifulSoup(response.content, "html.parser")
            selectors = site_selectors.get(domain, [])

        # 공통 선택자도 순서대로 시도
        selectors = selectors + common_selectors

        article_tag = None

        for selector in dict.fromkeys(selectors):
            candidate = soup.select_one(selector)

            if candidate and candidate.get_text(" ", strip=True):
                article_tag = candidate
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 이미지와 스크립트 제거
        for tag in article_tag.select("img, script, style, iframe"):
            tag.decompose()

        content = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        ).strip()

        if not content:
            result["error"] = "이미지 제거 후 본문이 비어 있습니다."
            return result

        result["markdown"] = content

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except UnicodeDecodeError:
        result["error"] = "문자 인코딩 확인 필요"

    except Exception as error:
        result["error"] = f"변환 실패: {type(error).__name__}"

    return result

In [ ]:
# 앞에서 저장한 기사 후보 파일
candidates_path = Path(
    "../private_data/innisfree_candidates.jsonl"
)

# 성공한 기사를 중간 저장할 파일
checkpoint_path = Path(
    "../private_data/innisfree_news_checkpoint.jsonl"
)

# 실패한 기사 주소와 이유를 기록할 파일
errors_path = Path(
    "../private_data/innisfree_news_errors.jsonl"
)


def read_jsonl(path):
    """JSONL 파일이 있으면 읽고, 없으면 빈 리스트를 반환합니다."""
    if not path.exists():
        return []

    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def append_jsonl(path, record):
    """JSON 객체 한 건을 파일에 추가합니다."""
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


# 후보, 기존 성공 결과, 기존 실패 기록 읽기
candidates = read_jsonl(candidates_path)
saved_articles = read_jsonl(checkpoint_path)
failed_articles = read_jsonl(errors_path)

# 이미 성공했거나 실패한 기사는 다시 요청하지 않음
processed_urls = {
    record["original_url"]
    for record in saved_articles + failed_articles
}

remaining = [
    record
    for record in candidates
    if record["original_url"] not in processed_urls
]

print("전체 후보:", len(candidates), "건")
print("이전에 저장한 기사:", len(saved_articles), "건")
print("이전에 제외한 기사:", len(failed_articles), "건")
print("이번에 처리할 기사:", len(remaining), "건")

success_count = 0
failure_count = 0

for number, candidate in enumerate(remaining, start=1):
    article_url = candidate["original_url"]
    title = candidate["title"]

    try:
        result = get_article_markdown(article_url)

        if result["markdown"]:
            # 기사 주소를 이용해 고유 ID 생성
            url_hash = hashlib.sha256(
                article_url.encode("utf-8")
            ).hexdigest()[:16]

            # 최종 뉴스 스키마
            record = {
                "document_id": f"NEWS-{url_hash}",
                "title": title,
                "original_url": article_url,
                "published_at": candidate["naver_provided_at"],
                "content": result["markdown"],
                "company_ids": ["COMPANY-INNISFREE"],
            }

            # 성공한 기사 한 건을 바로 저장
            append_jsonl(checkpoint_path, record)
            success_count += 1

        else:
            # 본문을 찾지 못한 기사는 오류 파일에만 기록
            append_jsonl(
                errors_path,
                {
                    "title": title,
                    "original_url": article_url,
                    "error": result["error"],
                },
            )
            failure_count += 1

    except Exception as error:
        append_jsonl(
            errors_path,
            {
                "title": title,
                "original_url": article_url,
                "error": f"{type(error).__name__}: {error}",
            },
        )
        failure_count += 1

    if number % 20 == 0:
        print(
            f"{number}/{len(remaining)}건 처리 · "
            f"저장 {success_count}건 · 제외 {failure_count}건"
        )

    # 언론사에 연속 요청을 보내지 않도록 대기
    time.sleep(1)

print("\n이번 작업 완료")
print("이번에 저장한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("중간 저장 파일:", checkpoint_path)

In [ ]:
import json
from pathlib import Path

# 기존 뉴스 파일과 이니스프리 수집 결과 파일
news_path = Path("../data/collected/anna/news.jsonl")
innisfree_path = Path("../private_data/innisfree_news_checkpoint.jsonl")

# JSONL 한 줄을 기사 하나의 딕셔너리로 읽기
def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

existing_news = read_jsonl(news_path)
innisfree_news = read_jsonl(innisfree_path)

# 기존 기사들의 URL 모으기
existing_urls = {article["original_url"] for article in existing_news}

# 이니스프리 기사 중 이미 있는 것과 새로 추가할 것 나누기
overlap = [
    article for article in innisfree_news
    if article["original_url"] in existing_urls
]

new_articles = [
    article for article in innisfree_news
    if article["original_url"] not in existing_urls
]

print("기존 뉴스:", len(existing_news), "건")
print("이니스프리 수집 뉴스:", len(innisfree_news), "건")
print("겹치는 기사:", len(overlap), "건")
print("새로 추가할 기사:", len(new_articles), "건")

In [ ]:
from copy import deepcopy

# 기존 목록을 복사해서 합칠 목록을 만들기
merged_news = deepcopy(existing_news)

# URL로 기존 기사를 빠르게 찾을 수 있도록 딕셔너리 만들기
news_by_url = {
    article["original_url"]: article
    for article in merged_news
}

added_count = 0
linked_count = 0

for article in innisfree_news:
    url = article["original_url"]

    if url in news_by_url:
        # 이미 있는 기사라면 이니스프리 ID를 추가
        existing_article = news_by_url[url]
        company_ids = existing_article.get("company_ids", [])

        if "COMPANY-INNISFREE" not in company_ids:
            company_ids.append("COMPANY-INNISFREE")
            existing_article["company_ids"] = company_ids
            linked_count += 1

    else:
        # 기존에 없는 기사라면 목록에 추가
        new_article = deepcopy(article)
        merged_news.append(new_article)
        news_by_url[url] = new_article
        added_count += 1

print("새로 추가한 기사:", added_count, "건")
print("기존 기사에 이니스프리 ID를 추가한 건수:", linked_count, "건")
print("합친 전체 기사:", len(merged_news), "건")

In [ ]:
import os
import tempfile
from datetime import datetime

# 1. 처음 읽은 뒤 기존 파일이 바뀌지 않았는지 확인
current_news = read_jsonl(news_path)

if current_news != existing_news:
    raise ValueError(
        "처음 읽은 뒤 news.jsonl이 변경됐어요. "
        "파일을 다시 읽고 병합을 다시 진행해 주세요."
    )

# 2. 기존 파일을 백업하기
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = news_path.with_name(f"news_backup_{timestamp}.jsonl")

with news_path.open("rb") as original:
    with backup_path.open("xb") as backup:
        backup.write(original.read())

# 3. 합친 결과를 임시 JSONL 파일에 쓰기
temp_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=news_path.parent,
        delete=False,
    ) as temp_file:
        temp_path = Path(temp_file.name)

        for article in merged_news:
            temp_file.write(json.dumps(article, ensure_ascii=False) + "\n")

    # 4. 임시 파일을 news.jsonl로 교체하기
    os.replace(temp_path, news_path)

finally:
    if temp_path is not None and temp_path.exists():
        temp_path.unlink()

# 5. 저장된 파일을 다시 읽어 확인하기
saved_news = read_jsonl(news_path)

print("백업 파일:", backup_path)
print("저장된 기사:", len(saved_news), "건")
print("기대 기사 수와 일치:", len(saved_news) == 2268)

In [ ]:
# 저장된 기사 중 기업 ID별 건수 확인
amore_count = 0
innisfree_count = 0
both_count = 0

for article in saved_news:
    company_ids = article.get("company_ids", [])

    has_amore = "COMPANY-AMOREPACIFIC" in company_ids
    has_innisfree = "COMPANY-INNISFREE" in company_ids

    if has_amore:
        amore_count += 1

    if has_innisfree:
        innisfree_count += 1

    if has_amore and has_innisfree:
        both_count += 1

print("아모레퍼시픽 ID가 있는 기사:", amore_count, "건")
print("이니스프리 ID가 있는 기사:", innisfree_count, "건")
print("두 기업 ID가 모두 있는 기사:", both_count, "건")

In [ ]:
import os
import requests
from dotenv import load_dotenv

# .env에서 네이버 API 키 불러오기
load_dotenv("../.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

if not naver_client_id or not naver_client_secret:
    raise ValueError("네이버 API 키를 찾을 수 없어요.")

# API 요청에 사용할 인증 정보
headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

# 에뛰드 최신 뉴스 검색
response = requests.get(
    "https://naverapihub.apigw.ntruss.com/search/v1/news",
    headers=headers,
    params={
        "query": "에뛰드",
        "display": 100,
        "start": 1,
        "sort": "date",
    },
    timeout=30,
)

response.raise_for_status()
etude_data = response.json()

print("네이버 검색 결과 전체:", etude_data.get("total"), "건")
print("이번에 가져온 검색 결과:", len(etude_data.get("items", [])), "건")

In [ ]:
import time
import pandas as pd

# 올해 연도 확인
current_year = pd.Timestamp.now(tz="Asia/Seoul").year

# 검색어, 정렬 방식, 가져올 페이지 수
etude_queries = [
    ("에뛰드", "date", 10),
]

# 2023년부터 올해까지 연도별 검색어 추가
for year in range(2023, current_year + 1):
    etude_queries.append((f"에뛰드 {year}", "sim", 3))

# 검색 결과를 담을 목록
etude_items = []

for query, sort, max_pages in etude_queries:
    print(f"\n검색 중: {query}")

    for page in range(max_pages):
        start = 1 + page * 100

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params={
                "query": query,
                "display": 100,
                "start": start,
                "sort": sort,
            },
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        etude_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        # 결과가 100건보다 적으면 다음 페이지가 없음
        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n가져온 검색 결과: {len(etude_items)}건")

In [ ]:
import re
import html
import pandas as pd

# 검색 결과에 들어 있는 HTML 태그와 특수문자 정리
def clean_news_text(text):
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 검색 결과를 표로 만들기
etude_candidates_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in etude_items
])

# 네이버에서 받은 게시 시각을 한국 시간으로 변환
etude_candidates_df["naver_provided_at"] = pd.to_datetime(
    etude_candidates_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# 2023년 1월 1일부터 현재까지
start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

# 기간에 들어가고 원문 URL이 있는 기사만 남기기
etude_candidates_df = etude_candidates_df[
    etude_candidates_df["naver_provided_at"].between(start_date, end_date)
    & etude_candidates_df["original_url"].notna()
    & etude_candidates_df["original_url"].str.strip().ne("")
].copy()

# 원문 URL이 같은 검색 결과는 하나만 남기기
etude_candidates_df = etude_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print("기간 확인과 중복 제거 후 기사 후보:", len(etude_candidates_df), "건")

# 앞의 5건을 살펴보기
display(etude_candidates_df.head())

In [ ]:
from pathlib import Path

etude_candidates_path = Path(
    "../private_data/etude_candidates.jsonl"
)

# 이미 같은 파일이 있으면 덮어쓰지 않기
if etude_candidates_path.exists():
    raise FileExistsError(
        f"이미 후보 파일이 있어요: {etude_candidates_path}"
    )

# 후보 표를 JSONL로 저장
with etude_candidates_path.open("x", encoding="utf-8") as file:
    file.write(
        etude_candidates_df.to_json(
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    )

print("저장한 기사 후보:", len(etude_candidates_df), "건")
print("저장 위치:", etude_candidates_path)

In [ ]:
import json

# 이니스프리 수집 때 만든 본문 추출 함수가 메모리에 있는지 확인
if "get_article_markdown" not in globals():
    raise ValueError(
        "본문 추출 함수가 없어요. "
        "노트북에서 get_article_markdown 함수를 정의한 셀을 먼저 실행해 주세요."
    )

# 저장해 둔 에뛰드 후보 파일 읽기
with etude_candidates_path.open(encoding="utf-8") as file:
    etude_candidates = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("읽어온 후보:", len(etude_candidates), "건")

In [ ]:
import time

etude_test_results = []

for candidate in etude_candidates[:5]:
    result = get_article_markdown(candidate["original_url"])

    etude_test_results.append({
        "title": candidate["title"],
        "original_url": candidate["original_url"],
        "content": result["markdown"],
        "error": result["error"],
    })

    if result["markdown"]:
        print("본문 후보 찾음:", candidate["title"])
    else:
        print("수집 실패:", candidate["title"], "-", result["error"])

    time.sleep(1)

print("\n시험한 기사:", len(etude_test_results), "건")

In [ ]:
import hashlib
import json
import time
from pathlib import Path

# 성공한 기사와 실패 기록을 저장할 파일
etude_checkpoint_path = Path(
    "../private_data/etude_news_checkpoint.jsonl"
)
etude_errors_path = Path(
    "../private_data/etude_news_errors.jsonl"
)


def read_saved_jsonl(path):
    """파일이 있으면 읽고, 없으면 빈 목록을 반환해요."""
    if not path.exists():
        return []

    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def append_jsonl(path, record):
    """JSONL 파일에 한 건씩 추가해요."""
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


# 이미 처리한 기사 확인: 중간에 멈춰도 이어서 실행할 수 있어요
saved_articles = read_saved_jsonl(etude_checkpoint_path)
failed_articles = read_saved_jsonl(etude_errors_path)

processed_urls = {
    article["original_url"]
    for article in saved_articles + failed_articles
}

remaining = [
    candidate for candidate in etude_candidates
    if candidate["original_url"] not in processed_urls
]

# 방금 시험한 5건의 결과를 URL로 찾아 쓸 수 있게 정리
test_by_url = {
    result["original_url"]: result
    for result in etude_test_results
}

print("전체 후보:", len(etude_candidates), "건")
print("이미 성공한 기사:", len(saved_articles), "건")
print("이미 실패한 기사:", len(failed_articles), "건")
print("이번에 처리할 기사:", len(remaining), "건")

success_count = 0
failure_count = 0

for number, candidate in enumerate(remaining, start=1):
    url = candidate["original_url"]

    try:
        if url in test_by_url:
            # 시험한 기사는 다시 요청하지 않고 결과 사용
            test_result = test_by_url[url]
            content = test_result["content"]
            error = test_result["error"]
        else:
            # 나머지 기사는 원문 페이지에서 본문 가져오기
            result = get_article_markdown(url)
            content = result["markdown"]
            error = result["error"]

            # 언론사에 요청을 연속으로 보내지 않도록 대기
            time.sleep(1)

        if content:
            # URL을 이용해 기사 ID 만들기
            url_hash = hashlib.sha256(
                url.encode("utf-8")
            ).hexdigest()[:16]

            record = {
                "document_id": f"NEWS-{url_hash}",
                "title": candidate["title"],
                "original_url": url,
                "published_at": candidate["naver_provided_at"],
                "content": content,
                "company_ids": ["COMPANY-ETUDE"],
            }

            append_jsonl(etude_checkpoint_path, record)
            success_count += 1

        else:
            # 실패한 기사는 본문에 넣지 않고 이유만 기록
            append_jsonl(etude_errors_path, {
                "title": candidate["title"],
                "original_url": url,
                "error": error,
            })
            failure_count += 1

    except Exception as error:
        append_jsonl(etude_errors_path, {
            "title": candidate["title"],
            "original_url": url,
            "error": f"{type(error).__name__}: {error}",
        })
        failure_count += 1

    if number % 20 == 0:
        print(
            f"{number}/{len(remaining)}건 처리 · "
            f"성공 {success_count}건 · 실패 {failure_count}건",
            flush=True,
        )

print("\n이번 작업 완료")
print("이번에 수집한 기사:", success_count, "건")
print("이번에 실패한 기사:", failure_count, "건")
print("성공 기사 저장 위치:", etude_checkpoint_path)
print("실패 기록 저장 위치:", etude_errors_path)

In [ ]:
import json
from copy import deepcopy
from pathlib import Path

news_path = Path("../data/collected/anna/news.jsonl")
etude_path = Path("../private_data/etude_news_checkpoint.jsonl")

# JSONL 파일 읽기
def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

existing_news = read_jsonl(news_path)
etude_news = read_jsonl(etude_path)

# 기존 기사를 복사하고 URL로 찾을 수 있게 정리
merged_news = deepcopy(existing_news)
news_by_url = {
    article["original_url"]: article
    for article in merged_news
}

added_count = 0
linked_count = 0

for article in etude_news:
    url = article["original_url"]

    if url in news_by_url:
        # 이미 있는 기사에는 에뛰드 ID 추가
        existing_article = news_by_url[url]
        company_ids = existing_article.get("company_ids") or []

        if "COMPANY-ETUDE" not in company_ids:
            company_ids.append("COMPANY-ETUDE")
            existing_article["company_ids"] = company_ids
            linked_count += 1

    else:
        # 기존에 없는 기사는 새로 추가
        new_article = deepcopy(article)
        merged_news.append(new_article)
        news_by_url[url] = new_article
        added_count += 1

print("새로 추가한 기사:", added_count, "건")
print("기존 기사에 에뛰드 ID를 추가한 건수:", linked_count, "건")
print("합친 전체 기사:", len(merged_news), "건")

In [ ]:
import os
import tempfile
from datetime import datetime

# 처음 읽은 뒤 기존 파일이 바뀌지 않았는지 확인
current_news = read_jsonl(news_path)

if current_news != existing_news:
    raise ValueError(
        "처음 읽은 뒤 news.jsonl이 변경됐어요. "
        "파일을 다시 읽고 병합을 다시 진행해 주세요."
    )

# 기존 뉴스 파일 백업
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = news_path.with_name(f"news_backup_{timestamp}.jsonl")

with news_path.open("rb") as original:
    with backup_path.open("xb") as backup:
        backup.write(original.read())

# 합친 기사를 임시 파일에 쓴 뒤 news.jsonl로 교체
temp_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=news_path.parent,
        delete=False,
    ) as temp_file:
        temp_path = Path(temp_file.name)

        for article in merged_news:
            temp_file.write(json.dumps(article, ensure_ascii=False) + "\n")

    os.replace(temp_path, news_path)

finally:
    if temp_path is not None and temp_path.exists():
        temp_path.unlink()

# 저장된 결과 다시 읽기
saved_news = read_jsonl(news_path)

print("백업 파일:", backup_path)
print("저장된 기사:", len(saved_news), "건")
print("기대 기사 수와 일치:", len(saved_news) == 2978)

In [ ]:
etude_count = 0

for article in saved_news:
    company_ids = article.get("company_ids", [])

    if "COMPANY-ETUDE" in company_ids:
        etude_count += 1

print("에뛰드 ID가 있는 기사:", etude_count, "건")

In [ ]:
import html
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests

# 검색어와 저장할 파일 이름
companies = {
    "에스쁘아": "espoir",
    "퍼시픽테크": "pacifictech",
    "아모스프로페셔널": "amosprofessional",
    "코스알엑스": "cosrx",
}

output_dir = Path("../private_data")
current_year = pd.Timestamp.now(tz="Asia/Seoul").year

# 이 셀을 실행하기 전에 네이버 API 인증 정보가 준비되어 있어야 해요.
if "headers" not in globals():
    raise ValueError("먼저 네이버 API 인증 정보를 준비한 셀을 실행해 주세요.")


def clean_title(text):
    """검색 결과의 HTML 태그와 특수문자를 정리해요."""
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


def collect_candidates(company_name, file_name):
    """한 회사의 검색 결과를 모아 후보 파일로 저장해요."""
    output_path = output_dir / f"{file_name}_candidates.jsonl"

    # 이미 만든 파일은 덮어쓰지 않아요.
    if output_path.exists():
        return company_name, f"기존 파일이 있어 건너뜀: {output_path}"

    queries = [(company_name, "date", 10)]

    for year in range(2023, current_year + 1):
        queries.append((f"{company_name} {year}", "sim", 3))

    items = []

    for query, sort, max_pages in queries:
        for page in range(max_pages):
            response = requests.get(
                "https://naverapihub.apigw.ntruss.com/search/v1/news",
                headers=headers,
                params={
                    "query": query,
                    "display": 100,
                    "start": 1 + page * 100,
                    "sort": sort,
                },
                timeout=30,
            )
            response.raise_for_status()

            page_items = response.json().get("items", [])
            items.extend(page_items)

            if len(page_items) < 100:
                break

            time.sleep(1)

    # 제목, URL, 게시일만 정리
    candidates = pd.DataFrame([
        {
            "title": clean_title(item.get("title")),
            "original_url": item.get("originallink"),
            "naver_provided_at": item.get("pubDate"),
        }
        for item in items
    ], columns=["title", "original_url", "naver_provided_at"])

    candidates["naver_provided_at"] = pd.to_datetime(
        candidates["naver_provided_at"],
        errors="coerce",
        utc=True,
    ).dt.tz_convert("Asia/Seoul")

    start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
    end_date = pd.Timestamp.now(tz="Asia/Seoul")

    candidates = candidates[
        candidates["naver_provided_at"].between(start_date, end_date)
        & candidates["original_url"].notna()
        & candidates["original_url"].str.strip().ne("")
    ].copy()

    candidates = candidates.drop_duplicates(
        subset=["original_url"]
    ).reset_index(drop=True)

    # 회사마다 별도의 파일에 저장
    with output_path.open("x", encoding="utf-8") as file:
        file.write(
            candidates.to_json(
                orient="records",
                lines=True,
                force_ascii=False,
                date_format="iso",
            )
        )

    return company_name, f"후보 {len(candidates)}건 저장: {output_path}"


# 네 회사의 검색 작업을 동시에 시작
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(collect_candidates, name, file_name): name
        for name, file_name in companies.items()
    }

    # 먼저 끝난 회사부터 결과 출력
    for future in as_completed(futures):
        name = futures[future]

        try:
            company_name, result = future.result()
            print(f"{company_name}: {result}")
        except Exception as error:
            print(f"{name}: 검색 실패 — {type(error).__name__}: {error}")

In [ ]:
import hashlib
import json
import threading
import time

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse


# 이니스프리·에뛰드 수집 때 만든 함수가 필요해요.
if "get_article_markdown" not in globals():
    raise ValueError(
        "get_article_markdown 함수가 없어요. "
        "본문 추출 함수를 정의한 셀을 먼저 실행해 주세요."
    )


# 회사마다 후보 파일과 기업 ID를 연결
company_settings = {
    "에스쁘아": {
        "file_name": "espoir",
        "company_id": "COMPANY-ESPOIR",
    },
    "퍼시픽테크": {
        "file_name": "pacifictech",
        "company_id": "COMPANY-PACIFICTECH",
    },
    "아모스프로페셔널": {
        "file_name": "amosprofessional",
        "company_id": "COMPANY-AMOSPROFESSIONAL",
    },
    "코스알엑스": {
        "file_name": "cosrx",
        "company_id": "COMPANY-COSRX",
    },
}

data_dir = Path("../private_data")


def read_jsonl_file(path):
    """JSONL 파일을 읽어요. 파일이 없으면 빈 목록을 반환해요."""
    if not path.exists():
        return []

    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def append_jsonl_file(path, record):
    """JSONL 파일에 한 건씩 추가해요."""
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


# 아직 처리하지 않은 후보를 URL별로 모아요.
# 하나의 URL이 여러 회사 검색에 나왔을 수도 있어요.
pending_by_url = defaultdict(list)

for company_name, settings in company_settings.items():
    file_name = settings["file_name"]

    candidates_path = data_dir / f"{file_name}_candidates.jsonl"
    checkpoint_path = data_dir / f"{file_name}_news_checkpoint.jsonl"
    errors_path = data_dir / f"{file_name}_news_errors.jsonl"

    if not candidates_path.exists():
        raise FileNotFoundError(f"후보 파일이 없어요: {candidates_path}")

    candidates = read_jsonl_file(candidates_path)
    saved = read_jsonl_file(checkpoint_path)
    failed = read_jsonl_file(errors_path)

    # 성공 또는 실패가 기록된 URL은 다시 요청하지 않아요.
    processed_urls = {
        record["original_url"]
        for record in saved + failed
    }

    remaining = [
        candidate
        for candidate in candidates
        if candidate["original_url"] not in processed_urls
    ]

    print(
        f"{company_name}: 후보 {len(candidates)}건 / "
        f"이미 처리 {len(processed_urls)}건 / "
        f"남은 후보 {len(remaining)}건"
    )

    for candidate in remaining:
        pending_by_url[candidate["original_url"]].append({
            "company_name": company_name,
            "company_id": settings["company_id"],
            "candidate": candidate,
            "checkpoint_path": checkpoint_path,
            "errors_path": errors_path,
        })


print("\n이번에 요청할 서로 다른 URL:", len(pending_by_url), "건")


# 같은 언론사에 대한 요청이 동시에 실행되지 않도록 잠금 만들기
host_locks = defaultdict(threading.Lock)


def collect_one_url(url):
    """URL 하나에서 본문을 가져와요."""
    host = urlparse(url).netloc.lower()

    # 같은 원문 주소의 도메인에는 한 번에 하나씩 요청해요.
    with host_locks[host]:
        try:
            return get_article_markdown(url)
        except Exception as error:
            return {
                "markdown": None,
                "error": f"{type(error).__name__}: {error}",
            }
        finally:
            # 같은 도메인의 다음 요청까지 잠시 대기
            time.sleep(1)


success_count = 0
failure_count = 0
completed_count = 0

# 동시에 최대 4개 URL 처리
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(collect_one_url, url): url
        for url in pending_by_url
    }

    for future in as_completed(futures):
        url = futures[future]
        result = future.result()
        completed_count += 1

        # 같은 URL이 여러 회사에 포함됐다면 회사별 파일에 각각 기록
        for job in pending_by_url[url]:
            candidate = job["candidate"]

            if result["markdown"]:
                url_hash = hashlib.sha256(
                    url.encode("utf-8")
                ).hexdigest()[:16]

                article = {
                    "document_id": f"NEWS-{url_hash}",
                    "title": candidate["title"],
                    "original_url": url,
                    "published_at": candidate["naver_provided_at"],
                    "content": result["markdown"],
                    "company_ids": [job["company_id"]],
                }

                append_jsonl_file(job["checkpoint_path"], article)
                success_count += 1

            else:
                failure = {
                    "title": candidate["title"],
                    "original_url": url,
                    "error": result["error"],
                }

                append_jsonl_file(job["errors_path"], failure)
                failure_count += 1

        if completed_count % 20 == 0 or completed_count == len(pending_by_url):
            print(
                f"URL {completed_count}/{len(pending_by_url)}건 처리 · "
                f"회사별 성공 기록 {success_count}건 · "
                f"회사별 실패 기록 {failure_count}건",
                flush=True,
            )

print("\n수집 완료!")
print("회사별 성공 기록:", success_count, "건")
print("회사별 실패 기록:", failure_count, "건")

In [ ]:
import json
from pathlib import Path

pacifictech_path = Path(
    "../private_data/pacifictech_news_checkpoint.jsonl"
)

with pacifictech_path.open(encoding="utf-8") as file:
    pacifictech_news = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

# 제목과 본문에 '퍼시픽테크'가 없는 기사 3건 살펴보기
no_name = [
    article for article in pacifictech_news
    if "퍼시픽테크" not in (
        article.get("title", "") + article.get("content", "")
    )
]

for article in no_name[:3]:
    print("제목:", article["title"])
    print("URL:", article["original_url"])
    print("본문 앞부분:", article["content"][:400])
    print("-" * 60)

In [ ]:
print(no_name[0]["content"])

In [ ]:
# 제목이나 본문에 '퍼시픽테크'가 들어간 기사
matched = [
    article for article in pacifictech_news
    if "퍼시픽테크" in (
        article.get("title", "") + article.get("content", "")
    )
]

print("이름이 포함된 기사:", len(matched), "건")

for article in matched[:5]:
    print("제목:", article["title"])
    print("URL:", article["original_url"])
    print("-" * 60)

In [ ]:
import json
from pathlib import Path

# 회사 이름과 수집 결과 파일
companies = {
    "에스쁘아": "espoir",
    "아모스프로페셔널": "amosprofessional",
    "코스알엑스": "cosrx",
}

data_dir = Path("../private_data")

for company_name, file_name in companies.items():
    # 본문 수집에 성공한 기사 파일
    checkpoint_path = data_dir / f"{file_name}_news_checkpoint.jsonl"

    with checkpoint_path.open(encoding="utf-8") as file:
        articles = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

    # 제목 또는 본문에 회사 이름이 있는 기사
    matched = [
        article for article in articles
        if company_name in (
            article.get("title", "") + article.get("content", "")
        )
    ]

    # 제목과 본문 모두에 회사 이름이 없는 기사
    not_matched = [
        article for article in articles
        if company_name not in (
            article.get("title", "") + article.get("content", "")
        )
    ]

    print(f"\n===== {company_name} =====")
    print("본문을 가져온 기사:", len(articles), "건")
    print("회사 이름이 있는 기사:", len(matched), "건")
    print("회사 이름이 없는 기사:", len(not_matched), "건")

    # 이름이 없는 기사 중 앞에서 2건 보여주기
    print("\n[이름이 없는 기사 예시]")

    for article in not_matched[:2]:
        print("제목:", article.get("title"))
        print("URL:", article.get("original_url"))
        print("본문 앞부분:", article.get("content", "")[:300])
        print("-" * 60)

In [ ]:
import json
from copy import deepcopy
from pathlib import Path

data_dir = Path("../private_data")
news_path = Path("../data/collected/anna/news.jsonl")

# 이번에 합칠 회사와 수집 결과 파일
companies = {
    "에스쁘아": "espoir",
    "아모스프로페셔널": "amosprofessional",
    "코스알엑스": "cosrx",
}


def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


# 기존 뉴스 읽기
existing_news = read_jsonl(news_path)
merged_news = deepcopy(existing_news)

# URL로 기사를 찾기 위한 딕셔너리
news_by_url = {
    article["original_url"]: article
    for article in merged_news
}

print("기존 기사:", len(existing_news), "건")

for company_name, file_name in companies.items():
    checkpoint_path = data_dir / f"{file_name}_news_checkpoint.jsonl"
    company_news = read_jsonl(checkpoint_path)

    added_count = 0
    linked_count = 0

    for article in company_news:
        url = article["original_url"]

        if url in news_by_url:
            # 이미 있는 기사: 기존 회사 ID를 유지하면서 새 ID 추가
            existing_article = news_by_url[url]
            company_ids = existing_article.get("company_ids") or []

            for company_id in article.get("company_ids", []):
                if company_id not in company_ids:
                    company_ids.append(company_id)
                    linked_count += 1

            existing_article["company_ids"] = company_ids

        else:
            # 기존에 없는 기사: 새로 추가
            new_article = deepcopy(article)
            merged_news.append(new_article)
            news_by_url[url] = new_article
            added_count += 1

    print(
        f"{company_name}: 새 기사 {added_count}건 / "
        f"기존 기사에 ID 추가 {linked_count}건"
    )

print("\n합친 전체 기사:", len(merged_news), "건")

In [ ]:
import json
from copy import deepcopy
from pathlib import Path

data_dir = Path("../private_data")
news_path = Path("../data/collected/anna/news.jsonl")

companies = {
    "에스쁘아": "espoir",
    "아모스프로페셔널": "amosprofessional",
    "코스알엑스": "cosrx",
}


def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


# 기존 뉴스 읽기
existing_news = read_jsonl(news_path)
merged_news = deepcopy(existing_news)

# 기존 기사를 URL로 찾기
news_by_url = {
    article["original_url"]: article
    for article in merged_news
}

print("기존 기사:", len(existing_news), "건")

for company_name, file_name in companies.items():
    checkpoint_path = data_dir / f"{file_name}_news_checkpoint.jsonl"
    company_news = read_jsonl(checkpoint_path)

    # 제목이나 본문에 회사 이름이 정확히 들어간 기사만 선택
    matched = [
        article for article in company_news
        if company_name in (
            (article.get("title") or "") + (article.get("content") or "")
        )
    ]

    added_count = 0
    linked_count = 0

    for article in matched:
        url = article["original_url"]

        if url in news_by_url:
            # 이미 있는 기사라면 회사 ID만 추가
            existing_article = news_by_url[url]
            company_ids = existing_article.get("company_ids") or []

            for company_id in article.get("company_ids", []):
                if company_id not in company_ids:
                    company_ids.append(company_id)
                    linked_count += 1

            existing_article["company_ids"] = company_ids

        else:
            # 기존에 없는 기사라면 새로 추가
            new_article = deepcopy(article)
            merged_news.append(new_article)
            news_by_url[url] = new_article
            added_count += 1

    print(f"\n{company_name}")
    print("본문 수집 기사:", len(company_news), "건")
    print("이름이 있는 기사:", len(matched), "건")
    print("이름이 없어 이번에 제외:", len(company_news) - len(matched), "건")
    print("새로 추가:", added_count, "건")
    print("기존 기사에 ID 추가:", linked_count, "건")

print("\n합친 전체 기사:", len(merged_news), "건")

In [ ]:
import os
import tempfile
from datetime import datetime

# 처음 읽은 뒤 기존 파일이 바뀌지 않았는지 확인
current_news = read_jsonl(news_path)

if current_news != existing_news:
    raise ValueError(
        "처음 읽은 뒤 news.jsonl이 변경됐어요. "
        "파일을 다시 읽고 병합을 다시 진행해 주세요."
    )

# 합친 결과 건수와 URL 중복 확인
merged_urls = [article["original_url"] for article in merged_news]

print("저장할 기사:", len(merged_news), "건")
print("URL 중복:", len(merged_urls) - len(set(merged_urls)), "건")

if len(merged_news) != 4580 or len(merged_urls) != len(set(merged_urls)):
    raise ValueError("기사 수 또는 URL 중복을 다시 확인해 주세요.")

# 기존 news.jsonl 백업
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = news_path.with_name(f"news_backup_{timestamp}.jsonl")

with news_path.open("rb") as original:
    with backup_path.open("xb") as backup:
        backup.write(original.read())

# 임시 파일에 쓴 후 기존 파일을 교체
temp_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=news_path.parent,
        delete=False,
    ) as temp_file:
        temp_path = Path(temp_file.name)

        for article in merged_news:
            temp_file.write(json.dumps(article, ensure_ascii=False) + "\n")

    # 임시 파일이 제대로 읽히는지 확인
    saved_temp = read_jsonl(temp_path)

    if len(saved_temp) != len(merged_news):
        raise ValueError("임시 파일의 기사 수가 맞지 않아요.")

    os.replace(temp_path, news_path)

finally:
    if temp_path is not None and temp_path.exists():
        temp_path.unlink()

# 저장 결과 확인
saved_news = read_jsonl(news_path)

print("\n백업 파일:", backup_path)
print("저장된 기사:", len(saved_news), "건")
print("기대 기사 수와 일치:", len(saved_news) == 4580)

In [ ]:
company_ids = {
    "에스쁘아": "COMPANY-ESPOIR",
    "아모스프로페셔널": "COMPANY-AMOSPROFESSIONAL",
    "코스알엑스": "COMPANY-COSRX",
}

for company_name, company_id in company_ids.items():
    count = sum(
        company_id in (article.get("company_ids") or [])
        for article in saved_news
    )

    print(f"{company_name} ID가 있는 기사: {count}건")